In [1]:
"""
Step 5b: crosstalk-strength sweep + control for the synthetic 5-qubit multiplexed benchmark.

Fixes vs step5_multiqubit_pipeline:
  * crosstalk strength s is a knob: 0.0 = no crosstalk (CONTROL), 1.0 = your original model, 2.0 = stronger
  * np.roll wrap-around removed (qubit 1 is no longer influenced by qubit 5)
  * CAL and CLF source shots come from DISJOINT random pools of the train file (no leakage, no label-order risk)
  * all qubits use the same window T_OPT = 500 (they are statistically identical copies of one qubit)
  * labels and source shots are drawn ONCE, so every strength and architecture is a paired comparison
  * paired standard errors on the gain of each architecture over single_lin4
Fixed-point + golden-vector export stay in your notebook (run those after you choose a strength).

Reminder: the data is SYNTHETIC. Summing 5 channels also costs SNR, so compare against the s=0 row,
not against the single-qubit 96.1% result.
"""
import os
import numpy as np

# ------------------------------------------------------------------ config
DATA_DIR = r"D:\MSU\Qubit_readout_FPGA\python"
N_QUBITS = 5
FS = 500.0e6
IF_FREQS = np.array([64.729e6, 25.366e6, -24.790e6, -70.269e6, -127.282e6])
T_BINS = 500
T_OPT = [500] * N_QUBITS
EMA_ALPHA = 0.05
RIDGE_LAMBDA_REL = 1e-4
CLIP = 99.99
N_CAL, N_CLF, N_TEST = 30_000, 50_000, 100_000
POOL_TRAIN_EACH = 100_000          # disjoint source-shot pools drawn from the train file (CAL / CLF)
STRENGTHS = [0.0, 0.5, 1.0, 2.0]
MODES = ["single_lin4", "multi_lin20", "multi_cross_quad"]
REF_1Q = 0.9610                    # Step 3 single-qubit result (lin4, T=500) for reference only
SEED = 42


def find_file(fname):
    for d in [DATA_DIR, os.path.join(DATA_DIR, "python"), "."]:
        cand = os.path.join(d, fname)
        if os.path.exists(cand):
            return cand
    raise FileNotFoundError(f"Cannot find {fname} in {DATA_DIR} or current folder.")


# ------------------------------------------------------------------ source pools (loaded once)
Xtr = np.load(find_file("0528_X_train_0_770.npy"), mmap_mode="r")
Xte = np.load(find_file("0528_X_test_0_770.npy"), mmap_mode="r")
ytr = np.load(find_file("0528_y_train_0_770.npy")).astype(int)
yte = np.load(find_file("0528_y_test_0_770.npy")).astype(int)

rng = np.random.default_rng(SEED)
perm = rng.permutation(len(ytr))
cal_pool = np.sort(perm[:POOL_TRAIN_EACH])
clf_pool = np.sort(perm[POOL_TRAIN_EACH:2 * POOL_TRAIN_EACH])
assert len(np.intersect1d(cal_pool, clf_pool)) == 0


def load_pool(X, idx):
    out = np.empty((len(idx), 2 * T_BINS), np.int16)
    for i in range(0, len(idx), 20_000):
        out[i:i + 20_000] = np.rint(X[idx[i:i + 20_000], :2 * T_BINS]).astype(np.int16)
    return out


print("loading source-shot pools ...")
S_cal, y_cal_pool = load_pool(Xtr, cal_pool), ytr[cal_pool]
S_clf, y_clf_pool = load_pool(Xtr, clf_pool), ytr[clf_pool]
S_te, y_te_pool = load_pool(Xte, np.arange(len(yte))), yte
print(f"pool label means (should be ~0.5): cal={y_cal_pool.mean():.3f} clf={y_clf_pool.mean():.3f} test={y_te_pool.mean():.3f}")

idx_cal = rng.integers(0, len(y_cal_pool), size=(N_CAL, N_QUBITS))
idx_clf = rng.integers(0, len(y_clf_pool), size=(N_CLF, N_QUBITS))
idx_te = rng.integers(0, len(y_te_pool), size=(N_TEST, N_QUBITS))
Y_cal, Y_clf, Y_te = y_cal_pool[idx_cal], y_clf_pool[idx_clf], y_te_pool[idx_te]

t_sec = np.arange(T_BINS, dtype=np.float64) / FS
carriers = np.exp(1j * 2.0 * np.pi * IF_FREQS[:, None] * t_sec[None, :])
LIN = np.array([[1.00, 0.15, 0.04, 0.01, 0.00],
                [0.15, 1.00, 0.16, 0.04, 0.01],
                [0.04, 0.16, 1.00, 0.14, 0.04],
                [0.01, 0.04, 0.14, 1.00, 0.15],
                [0.00, 0.01, 0.04, 0.15, 1.00]])
PH = np.array([[0.00, 0.28, 0.08, 0.00, 0.00],
               [-0.26, 0.00, 0.30, 0.08, 0.00],
               [0.07, -0.28, 0.00, 0.26, 0.07],
               [0.00, 0.07, -0.25, 0.00, 0.28],
               [0.00, 0.00, 0.07, -0.27, 0.00]])


def synth(S, idx, Y, s, chunk=5000):
    """Multiplex 5 source shots onto one bus with crosstalk strength s."""
    n = len(idx)
    L = np.eye(N_QUBITS) + s * (LIN - np.eye(N_QUBITS))
    P = s * PH
    I_out = np.empty((n, T_BINS), np.int16)
    Q_out = np.empty((n, T_BINS), np.int16)
    for a in range(0, n, chunk):
        b = min(a + chunk, n)
        bb = np.empty((b - a, N_QUBITS, T_BINS), np.complex128)
        for q in range(N_QUBITS):
            rows = S[idx[a:b, q]].astype(np.float64)
            bb[:, q, :] = rows[:, 0::2] + 1j * rows[:, 1::2]
        sgn = 2.0 * Y[a:b] - 1.0
        theta = sgn @ P.T
        prev = np.zeros_like(sgn)
        prev[:, 1:] = sgn[:, :-1]                      # neighbour q-1 only, no wrap-around
        pull = 1.0 + 0.12 * s * prev
        bb = bb * (pull[:, :, None] * np.exp(1j * theta[:, :, None]))
        bb = np.einsum("pq,bqt->bpt", L, bb)
        bus = np.einsum("bkt,kt->bt", bb, carriers) / np.sqrt(N_QUBITS)
        I_out[a:b] = np.clip(np.rint(bus.real), -32768, 32767).astype(np.int16)
        Q_out[a:b] = np.clip(np.rint(bus.imag), -32768, 32767).astype(np.int16)
    return I_out, Q_out


# ------------------------------------------------------------------ calibration + features (same as your notebook)
def ema_1d(x, alpha):
    out = np.empty_like(x)
    out[0] = x[0]
    for t in range(1, len(x)):
        out[t] = alpha * x[t] + (1.0 - alpha) * out[t - 1]
    return out


def calibrate(I_raw, Q_raw, Y):
    S_raw = I_raw.astype(np.float64) + 1j * Q_raw.astype(np.float64)
    W = np.zeros((N_QUBITS, 4, 2, T_BINS))
    for q in range(N_QUBITS):
        Zq = S_raw * np.exp(-1j * 2.0 * np.pi * IF_FREQS[q] * t_sec)[None, :]
        Iq, Qq, yq = Zq.real, Zq.imag, Y[:, q]
        m0, m1 = yq == 0, yq == 1
        mI0, mQ0, vI0, vQ0 = Iq[m0].mean(0), Qq[m0].mean(0), Iq[m0].var(0), Qq[m0].var(0)
        mI1, mQ1, vI1, vQ1 = Iq[m1].mean(0), Qq[m1].mean(0), Iq[m1].var(0), Qq[m1].var(0)
        mtv = np.c_[Iq.mean(1), Qq.mean(1)]
        C0, C1 = mtv[m0].mean(0), mtv[m1].mean(0)
        relax = m1 & (np.linalg.norm(mtv - C0, axis=1) < 0.5 * np.linalg.norm(C0 - C1))
        w_mf_I = ema_1d(mI1 - mI0, EMA_ALPHA) / (0.5 * (vI0 + vI1))
        w_mf_Q = ema_1d(mQ1 - mQ0, EMA_ALPHA) / (0.5 * (vQ0 + vQ1))
        if relax.sum() >= 20:
            mIR, mQR, vIR, vQR = Iq[relax].mean(0), Qq[relax].mean(0), Iq[relax].var(0), Qq[relax].var(0)
            w_rmf_I = ema_1d(mIR - mI0, EMA_ALPHA) / (0.5 * (vI0 + vIR))
            w_rmf_Q = ema_1d(mQR - mQ0, EMA_ALPHA) / (0.5 * (vQ0 + vQR))
        else:
            print(f"   warning: qubit {q + 1} has only {relax.sum()} relaxation traces; RMF disabled")
            w_rmf_I = np.zeros(T_BINS)
            w_rmf_Q = np.zeros(T_BINS)
        for w in (w_mf_I, w_mf_Q, w_rmf_I, w_rmf_Q):
            w[T_OPT[q]:] = 0.0
        c, sn = np.cos(2.0 * np.pi * IF_FREQS[q] * t_sec), np.sin(2.0 * np.pi * IF_FREQS[q] * t_sec)
        W[q, 0, 0], W[q, 0, 1] = w_mf_I * c, w_mf_I * sn
        W[q, 1, 0], W[q, 1, 1] = -w_mf_Q * sn, w_mf_Q * c
        W[q, 2, 0], W[q, 2, 1] = w_rmf_I * c, w_rmf_I * sn
        W[q, 3, 0], W[q, 3, 1] = -w_rmf_Q * sn, w_rmf_Q * c
    return W


def accumulate(I_raw, Q_raw, W, chunk=5000):
    N = len(I_raw)
    out = np.empty((N, N_QUBITS, 4))
    WI = W[:, :, 0, :].reshape(N_QUBITS * 4, T_BINS).T
    WQ = W[:, :, 1, :].reshape(N_QUBITS * 4, T_BINS).T
    for s in range(0, N, chunk):
        e = min(s + chunk, N)
        out[s:e] = (I_raw[s:e].astype(np.float64) @ WI + Q_raw[s:e].astype(np.float64) @ WQ).reshape(e - s, N_QUBITS, 4)
    return out


def build_features(V, q, mode):
    vk = V[:, q, :]
    if mode == "single_lin4":
        return vk
    lin_all = V.reshape(len(V), N_QUBITS * 4)
    if mode == "multi_lin20":
        return lin_all
    if mode == "multi_cross_quad":
        self_quad = [vk[:, i] * vk[:, j] for i in range(4) for j in range(i, 4)]
        cross = []
        for j in range(N_QUBITS):
            if j == q:
                continue
            vj = V[:, j, :]
            cross += [vk[:, 0] * vj[:, 0], vk[:, 0] * vj[:, 1], vk[:, 1] * vj[:, 0], vk[:, 1] * vj[:, 1]]
        return np.c_[lin_all, np.stack(self_quad, 1), np.stack(cross, 1)]
    raise ValueError(mode)


def best_threshold(s, y):
    order = np.argsort(s)
    ss, ys = s[order], y[order]
    n0 = (ys == 0).sum()
    c1 = np.r_[0, np.cumsum(ys == 1)]
    c0 = np.r_[0, np.cumsum(ys == 0)]
    k = int(np.argmin(c1 + (n0 - c0)))
    if k == 0:
        return ss[0] - 1.0
    if k == len(ss):
        return ss[-1]
    return 0.5 * (ss[k - 1] + ss[k])


def cross_fidelity(Pred, Ytrue):
    vals = []
    for j in range(N_QUBITS):
        for k in range(N_QUBITS):
            if j != k:
                p10 = np.mean(Pred[Ytrue[:, k] == 0, j] == 1)
                p01 = np.mean(Pred[Ytrue[:, k] == 1, j] == 0)
                vals.append(abs(1.0 - (p10 + p01)))
    return float(np.mean(vals))


# ------------------------------------------------------------------ sweep
summary = []
print(f"\nSingle-qubit reference from Step 3 (lin4, T=500): {REF_1Q:.4f}  (different SNR; compare against the s=0 row instead)")
for s in STRENGTHS:
    print(f"\n=== crosstalk strength s = {s} ({'CONTROL: no crosstalk' if s == 0 else 'x original model'}) ===")
    I_cal, Q_cal = synth(S_cal, idx_cal, Y_cal, s)
    I_clf, Q_clf = synth(S_clf, idx_clf, Y_clf, s)
    I_te, Q_te = synth(S_te, idx_te, Y_te, s)
    W = calibrate(I_cal, Q_cal, Y_cal)
    Ac, At = accumulate(I_clf, Q_clf, W), accumulate(I_te, Q_te, W)
    mu = Ac.mean(0)
    A = np.percentile(np.abs(Ac - mu), CLIP, axis=0)
    n = np.ceil(np.log2(np.maximum(A, 1e-9)))
    norm = lambda a: np.clip((a + 2.0 ** n - mu) / 2.0 ** (n + 1), 0, 1) - 0.5
    Vc, Vt = norm(Ac), norm(At)

    res = {}
    print(f"{'architecture':<18s} | " + " ".join(f"{'Q' + str(q + 1):>6s}" for q in range(N_QUBITS)) + " | mean F  <|F_CF|>")
    for mode in MODES:
        pred = np.zeros_like(Y_te)
        corr = np.zeros(Y_te.shape, bool)
        fids = []
        for q in range(N_QUBITS):
            Fc = np.c_[np.ones(len(Vc)), build_features(Vc, q, mode)]
            Ft = np.c_[np.ones(len(Vt)), build_features(Vt, q, mode)]
            w = np.linalg.solve(Fc.T @ Fc + RIDGE_LAMBDA_REL * len(Fc) * np.eye(Fc.shape[1]),
                                Fc.T @ (2.0 * Y_clf[:, q] - 1.0))
            thr = best_threshold(Fc @ w, Y_clf[:, q])
            pq = (Ft @ w) > thr
            pred[:, q] = pq
            corr[:, q] = pq == (Y_te[:, q] == 1)
            fids.append(0.5 * (np.mean(~pq[Y_te[:, q] == 0]) + np.mean(pq[Y_te[:, q] == 1])))
        res[mode] = dict(fids=fids, corr=corr, cf=cross_fidelity(pred, Y_te))
        print(f"{mode:<18s} | " + " ".join(f"{f:6.4f}" for f in fids) + f" | {np.mean(fids):.4f}  {res[mode]['cf']:.5f}")

    base = res["single_lin4"]["corr"].mean(1)
    row = dict(s=s, F_lin4=np.mean(res["single_lin4"]["fids"]), cf_lin4=res["single_lin4"]["cf"])
    for mode in ("multi_lin20", "multi_cross_quad"):
        d = res[mode]["corr"].mean(1) - base
        row[mode] = (100 * d.mean(), 100 * d.std() / np.sqrt(len(d)))
        row["F_" + mode] = np.mean(res[mode]["fids"])
        row["cf_" + mode] = res[mode]["cf"]
    summary.append(row)

print("\n=================== SUMMARY: gain over single_lin4 (fidelity points, +/- 1 standard error) ===================")
print(f"{'strength':>8s} | {'F single_lin4':>13s} | {'multi_lin20':>16s} | {'multi_cross_quad':>18s} | {'<|F_CF|> lin4 -> cross_quad':>28s}")
for r in summary:
    print(f"{r['s']:8.1f} | {r['F_lin4']:13.4f} | {r['multi_lin20'][0]:+7.2f}+/-{r['multi_lin20'][1]:.2f} | "
          f"{r['multi_cross_quad'][0]:+8.2f}+/-{r['multi_cross_quad'][1]:.2f} | {r['cf_lin4']:.5f} -> {r['cf_multi_cross_quad']:.5f}")
print("\nHow to read: gains should be ~0 at s=0 and grow with s. If they are large at s=0, something else is going on.")
print("Remember: this is a SYNTHETIC crosstalk model. It tests your pipeline, not real-chip crosstalk.")


loading source-shot pools ...
pool label means (should be ~0.5): cal=0.503 clf=0.501 test=0.500

Single-qubit reference from Step 3 (lin4, T=500): 0.9610  (different SNR; compare against the s=0 row instead)

=== crosstalk strength s = 0.0 (CONTROL: no crosstalk) ===
architecture       |     Q1     Q2     Q3     Q4     Q5 | mean F  <|F_CF|>
single_lin4        | 0.8648 0.8484 0.8723 0.8965 0.9274 | 0.8819  0.00344
multi_lin20        | 0.8651 0.8482 0.8721 0.8964 0.9274 | 0.8819  0.00378
multi_cross_quad   | 0.8644 0.8482 0.8726 0.8964 0.9270 | 0.8817  0.00371

=== crosstalk strength s = 0.5 (x original model) ===
architecture       |     Q1     Q2     Q3     Q4     Q5 | mean F  <|F_CF|>
single_lin4        | 0.8624 0.8398 0.8638 0.8898 0.9220 | 0.8756  0.01274
multi_lin20        | 0.8652 0.8473 0.8704 0.8957 0.9242 | 0.8805  0.00984
multi_cross_quad   | 0.8649 0.8490 0.8715 0.8969 0.9252 | 0.8815  0.00993

=== crosstalk strength s = 1.0 (x original model) ===
architecture       |     Q1 